# ConditionToCure
## Anzar Alvi

### Task overview
__Purpose__
- Evaluate skills in handling biological and clinical data using modern data engineering tools and practices
  
__Description__
- Work with biological data that requires integration, transformation, and analysis using various technologies commonly used in bioinformatics

__Technical requirements__
- Python 3.10 or higher
- Basic understanding of biological data formats and concepts
- RESTful services and working with web APIs
- Basic understanding of ontology, semantic formalism, and thesaurus such as MESH

__Goal__
- Create a data extraction protocol to retrieve information from clinicaltrials.gov

__Key output__
- Single function that accepts a str argument representing the "Conditiion or Disease", the exepcted output of the function is:
```
{
    "clinical_trials": [...],
    "indications": [...],
    "treatments": [...]
}
```


### Import libraries

In [5]:
import requests # Need to import this in order to make the "get" call to clinicaltrials.gov
import json # Need to import this in order to store the API call results into a JSON, and we also want to serialize our output

### Define Clinical_Trial object

In [8]:
class Clinical_Trial:
    """A single clinical trial object.

    Instance Attributes:
        - id: the unique identification code for the clinical trial study
        - title: official title for the clinical trial study
        - sponsor: name of the lead sponsor for the clinical trial study
        - overall_status: the overall status for the clinical trial study
        - last_known_status: the last known status for the clinical trial study
        - termination_reason: the reason why the clinical trial study stopped (if applicable)
        - phase: the phase for the clinical trial study
        - start_date: the start date for the clinical trial study
        - primary_completion_date: the primary completion date for the clinical trial study
        - completion_date: the completion date for the clinical trial study
        - last_update_date: the most recent date on which changes to a clinical trial study were
        posted on clinicaltrials.gov
            - Here we use the "Last Update Submitted Date" and not the "Last Update Posted Date"
            because we want to track official dates where changes to a study record were made
            available on clinicaltrials.gov (which is the posted date), the submitted date is
            not the date the changes are made apparent on clinicaltrials.gov
        - is_us_export: True if the drug/device is US FDA-regulated, and False if not
        - accepts_healthy_volunteers: True if the study is accepting healthy volunteers, and False if not
        - indication: the condition MeSH identification code(s) and term(s) for the clinical trial study
            - clinicaltrials.gov does not explicitly define "indication ID" and "indication name", and since
            conditions in the clinical trials are categorized with unique MeSH ID's, we can use "condition ID"
            (MeSH ID) and "condition name" (MeSH term) as proxies for "indication ID" and "indication name"
        - treatment: the intervention MeSH identification code(s) and term(s) for the clinical trial study
            - clinicaltrials.gov does not explicitly define "treatment ID" and "treatment name", and since
            interventions in the clinical trials are categorized with unique MeSH ID's, we can use "intervnetion ID"
            (MeSH ID) and "intervention name" (MeSH term) as proxies for "treatment ID" and "treatment name"
    """
    id: str
    title: str
    sponsor: str
    overall_status: str
    last_known_status: str
    termination_reason: str
    phase: str
    start_date: str # This is not type date because the formatting for dates on clinicaltrials.gov are inconsistent
    primary_completion_date: str # This is not type date because the formatting for dates on clinicaltrials.gov are inconsistent
    completion_date: str # This is not type date because the formatting for dates on clinicaltrials.gov are inconsistent
    last_update_date: str # This is not type date because the formatting for dates on clinicaltrials.gov are inconsistent
    is_us_export: bool
    accepts_healthy_volunteers: bool
    indication: list[str]
    treatment: list[str]

    def __init__(self, id: str, title: str, sponsor: str, overall_status: str, last_known_status: str, 
                 termination_reason: str, phase: str, start_date: str, primary_completion_date: str,
                 completion_date: str, last_update_date: str, is_us_export: bool, accepts_healthy_volunteers: bool,
                 indication: list[str], treatment: list[str]) -> None:
        """Initialize a new Clinical_Trial object with the given attributes.
        """
        self.id = id
        self.title = title
        self.sponsor = sponsor
        self.overall_status = overall_status
        self.last_known_status = last_known_status
        self.termination_reason = termination_reason
        self.phase = phase
        self.start_date = start_date
        self.primary_completion_date = primary_completion_date
        self.completion_date = completion_date
        self.last_update_date = last_update_date
        self.is_us_export = is_us_export
        self.accepts_healthy_volunteers = accepts_healthy_volunteers
        self.indication = indication
        self.treatment = treatment

### Justification for Clinical_Trial object information
__Required Information__
- id: allows for the user to search for the unique clinical trial study quickly
- title: allows for the user to get a brief overview of the study
- spnosor: allows for the user to understand which parties were involved in the clinical trial study,
  potentially for their own collaborations in the future
- overall_status: allows for the user to understand the progress of the clinical trial study,
  if many overall statuses are similar, user can gauge how fast the field of research they are involved in
  is moving
- last_known_status: allows for the user to understand the recency of the clinical trial study to understand how
  active this area of research is
- termination_reason: allows for the user to understand why a study ended, it's important to understand this as it can
  help users to effectively plan their own methodology with their research
- phase: helps users understand how far along a certain clinical trial study is
- start_date: allows for users to understand how recent/old a study is
- primary_completion_date: lets users understand how long it takes (approximately) for a study to be completed
- completion_date: this information coupled with the start_date gives users a great sense of the timeline
  for a clinical trial study
- last_update_date: if users want to know recent edits/changes made to a clinical trial study, this information is beneficial
- indication: gives users a holistic overview of the condition/indication names and MeSH codes for the condition/disease
- treatment: gives users a holistic overview of the intervention/treatment names and MeSH codes for the condition/disease

__Additional Information__
- is_us_export: very useful information for foregin governments/customers, if the user is aware about the product's US regulations/quality standards, this allows for a good
  understanding of safety and efficacy of the product
- accepts_healthy_volunteers: useful information for users who may want to contribute towards the clinical trial study/area of research

### Define Indication object

In [12]:
class Indication:
    """A single Indication object.

    Instance Attributes:
        - name: name of indication
        - code: code of indication
    """
    name: str
    code: str

    def __init__(self, name: str, code: str) -> None:
        """Initialize a new Indication object with the given attributes.
        """
        self.name = name
        self.code = code

### Justification for Indication object information
__Required Information__
- name: official MeSH name for the condition/disease is important for the user to know
- code: official MeSH ID/code is important for the user to know if they want to easily learn more about the condition

__No additional information needed to be included__

### Define Treatment object

In [16]:
class Treatment:
    """A single Treatment object.

    Instance Attributes:
        - name: name of treatment
        - code: code of treatment
    """
    name: str
    code: str

    def __init__(self, name: str, code: str) -> None:
        """Initialize a new Treatment object with the given attributes.
        """
        self.name = name
        self.code = code

### Justification for Treatment object information
__Required Information__
- name: official MeSH name for the treatment is important for the user to know
- code: official MeSH ID/code is important for the user to know if they want to easily learn more about the treatment

__No additional information needed to be included__

### Make API call, extract data, deal with missing data, deal with duplicate data, serialize results, and format the return nicely for ConditionToCure function

In [20]:
def extract_data(argument: str) -> list[list[dict]]:
    """This function will input the condition/disease query from the ConditionToCure function and outputs a list of 3 lists of dictionaries:
    1. List 1: Contains dictionaries, each dictionary is a serialized Clinical_Trial object
    2. List 2: Contains dictionaries, each dictionary is a serialized Indication object
    3. List 3: Contains dictionaries, each dictionary is a serialized Treatment object

    Near the bottom of the function body, a comment gives instructions on how to return un-seralized results.
    """
    url = f'https://clinicaltrials.gov/api/v2/studies' # This is the general URL to access clinicaltrials.gov
    params = {
        "pageSize": 50, # We are limiting to 50 results per page, but every page is being processed, so all 
                        # studies will be returned for a given condition/disease
        "query.cond": argument # Input from the client (condition/disease)
    }

    clinical_trials = [] # Initialize an empty list, it will be populated with Clinical_Trial objects
                         # We will work on the Indication and Treatment lists after populating clinical_trials
    while True: # We only break out of this loop when there is no next page (so all studies have been examined)
        response = requests.get(url, params) # Get the result from clinicaltrials.gov
        if response.status_code == 200: # Ensure that the get request is successful (code 200 indicates success)
            information = response.json() # Save the result into a JSON to easily access the needed information
            studies = information.get('studies') # Save the studies from the JSON response
            for study in studies: # Loop through all studies from the result
                # We need to use try/except statements here because if there is any missing information, an error will be raised,
                # which we will handle by setting the attribute to None, 'NA' or [], most studies contain the needed information, but for
                # the ones that do not, we need to deal with this
                try:
                    id = study['protocolSection']['identificationModule'].get('nctId') # Retrieve unique ID of clinical trial
                except:
                    id = 'NA'
                try:
                    title = study['protocolSection']['identificationModule'].get('officialTitle') # Retrieve clinical trial title
                except:
                    title = 'NA'
                try:
                    sponsor = study['protocolSection']['sponsorCollaboratorsModule']['leadSponsor'].get('name') # Retrieve name of sponsor
                except:
                    sponsor = 'NA'
                try:
                    overall_status = study['protocolSection']['statusModule'].get('overallStatus') # Retrieve overall status of trial
                except:
                    overall_status = 'NA'
                try:
                    last_known_status = study['protocolSection']['statusModule'].get('lastKnownStatus') # Retrieve last known status
                except:
                    last_known_status = 'NA'
                try:
                    termination_reason = study['protocolSection']['statusModule'].get('whyStopped') # Retrieve termination reason
                except:
                    termination_reason = 'NA'
                try:
                    phase = study['protocolSection']['designModule'].get('phases') # Retrieve phase
                except:
                    phase = 'NA'
                try:
                    start_date = study['protocolSection']['statusModule']['startDateStruct'].get('date') # Retrieve start date
                except:
                    start_date = 'NA'
                try:
                    primary_completion_date = study['protocolSection']['statusModule']['primaryCompletionDateStruct'].get('date') # Retrieve primary completion date
                except:
                    primary_completion_date = 'NA'
                try:
                    completion_date = study['protocolSection']['statusModule']['completionDateStruct'].get('date') # Retreive completion date
                except:
                    completion_date = 'NA'
                try:
                    last_update_date = study['protocolSection']['statusModule']['lastUpdatePostDateStruct'].get('date') # Retrieve last update date
                except:
                    last_update_date = 'NA'
                try:
                    is_us_export = study['protocolSection']['oversightModule'].get('isUsExport') # Retreive if there is US export
                except:
                    is_us_export = None # We set this as None and not 'NA' because this attribute is stored as a bool, not a string
                try:
                    accepts_healthy_volunteers = study['protocolSection']['eligibilityModule'].get('healthyVolunteers') # Retreive if healthy volunteers are accepted
                except:
                    accepts_healthy_volunteers = None # We set this as None and not 'NA' because this attribute is stored as a bool, not a string
                try:
                    # We don't index into 'meshes' indicating only one indication code and name
                    # See the next try statement which takes care of the case where there are multiple indication codes and names
                    indication_code = study['derivedSection']['conditionBrowseModule']['meshes'].get('id') # Retreive indication MeSH code
                    indication_name = study['derivedSection']['conditionBrowseModule']['meshes'].get('term') # Retrieve indication name
                    indication_str = indication_name + ": " + indication_code
                    indication_str_list = []
                    indication_str_list.append(indication_str) # Save indication code and name as a string like "[name]: [code]", save a list of strings
                except:
                    indication_str_list = []
                try:
                    indication_str_list = []
                    # When analyzing indication codes/names for a study, there were studies with multiple indications, so we must index into 'meshes',
                    # we handle this here by looping through all elements in mesh
                    length = len(study['derivedSection']['conditionBrowseModule']['meshes'])
                    for i in range(length):
                        indication_code = study['derivedSection']['conditionBrowseModule']['meshes'][i].get('id') # Retrieve indication MeSH code at index i of 'mesh'
                        indication_name = study['derivedSection']['conditionBrowseModule']['meshes'][i].get('term') # Retrieve indication MeSH name at index i of "mesh'
                        indication_str = indication_name + ": " + indication_code
                        indication_str_list.append(indication_str) # Save indication code and name as a string like "[name]: [code]", save a list of strings
                except:
                    indication_str_list = []
                try:
                    # We don't index into 'meshes' indicating only one treatment code and name
                    # See the next try statement which takes care of the case where there are multiple treatment codes and names
                    treatment_code = study['derivedSection']['interventionBrowseModule']['meshes'].get('id') # Retreive treatment MeSH code
                    treatment_name = study['derivedSection']['interventionBrowseModule']['meshes'].get('term') # Retrieve treatment name
                    treatment_str = treatment_name + ": " + treatment_code
                    treatment_str_list = []
                    treatment_str_list.append(treatment_str) # Save treatment code and name as a string like "[name]: [code]", save a list of strings
                except:
                    treatment_str_list = []
                try:
                    treatment_str_list = []
                    # When analyzing treatment codes/names for a study, there were studies with multiple treatments, so we must index into 'meshes',
                    # we handle this here by looping through all elements in mesh
                    length = len(study['derivedSection']['interventionBrowseModule']['meshes'])
                    for i in range(length):
                        treatment_code = study['derivedSection']['interventionBrowseModule']['meshes'][i].get('id') # Retrieve treatment MeSH code at index i of 'mesh'
                        treatment_name = study['derivedSection']['interventionBrowseModule']['meshes'][i].get('term') # Retrieve treatment MeSH name at index i of "mesh'
                        treatment_str = treatment_name + ": " + treatment_code
                        treatment_str_list.append(treatment_str) # Save treatment code and name as a string like "[name]: [code]", save a list of strings
                except:
                    treatment_str_list = []
                trial = Clinical_Trial(id, title, sponsor, overall_status, last_known_status, termination_reason, phase, start_date, 
                                       primary_completion_date, completion_date, last_update_date, is_us_export, accepts_healthy_volunteers,
                                       indication_str_list, treatment_str_list) # Create a clinical trial object
                clinical_trials.append(trial) # Append the clinical trial object to our clinical_trials list
            next_page_token = information.get('nextPageToken') # Retreive next_page_token
            if next_page_token:
                params['pageToken'] = next_page_token
            else:
                break # Exit the "while True" loop if there is no next_page_token
        else:
            print("Failed to get data from clinicaltrials.gov. Status code:", response.status_code) # Print this if the status code is not 200 (not a success)

    # This creates our list of Indication objects (indications) and list of Treatment objects (treatments)
    # Helper function is at the bottom of the notebook
    indication_and_treatment = create_indication_and_treatment(clinical_trials) # HELPER
    indications = indication_and_treatment[0]
    treatments = indication_and_treatment[1]

    # Remove duplicates
    # Helper function is at the bottom of the notebook
    indications = remove_duplicates(indications) # HELPER
    treatments = remove_duplicates(treatments) # HELPER

    # INSTRUCTIONS FOR RETURNING UNSERIALIZED RESULTS
        # 1. Uncomment the "return [clinical_trials, indications, treatments]" line
        # 2. Comment the serialization code
        # 3. Comment the final return statement in this function
    # return [clinical_trials, indications, treatments]

    # Serialize the Clinical_Trial objects in clinical_trials to JSON and save in a list
    # Serialize the Indication objects in indications to JSON and save in a list
    # Serialize the Treatment objects in treatments to JSON and save it in a list
    # Helper function is at the bottom of the notebook
    ser_clinical_trials = serialize(clinical_trials) # HELPER
    ser_indications = serialize(indications) # HELPER
    ser_treatments = serialize(treatments) # HELPER

    # Return serialized results
    return [ser_clinical_trials, ser_indications, ser_treatments]

### Quick note about try/except statements in the extract_data function
It may seem unnecessary to include try/except statements for every piece of information, some studies may even already have information like 'None' or 'NA'. However, it's very important to include because when analyzing the extracted JSON data from clinicaltrials.gov, some studies did not even include a key to access the information. This raises an error when trying to access it which needs to be handled.

### Final ConditionToCure function

In [11]:
def ConditionToCure(argument: str) -> dict[str, list]:
    """Key function of the program that accepts argument representing the "Condition or Disease" and
    produces an acceptable output as follows:

    {
      "clinical_trials": [...],
      "indications": [...],
      "treatments": [...]
    }

    The output is serialized.
    """
    information = extract_data(argument)
    result = {}
    result['clinical_trials'] = information[0] # First key of the expected output is "clinical_trials"
    result['indications'] = information[1] # Second key of the expected output is "indications"
    result['treatments'] = information[2] # Third key of the expected output is "treatments"
    return result

### Try the ConditionToCure function

In [34]:
ConditionToCure('fibrolamellar carcinoma')

{'clinical_trials': [{'id': 'NCT01642186',
   'title': 'A Randomized Three Arm Phase II Study of (1) Everolimus, (2) Estrogen Deprivation Therapy (EDT) With Leuprolide + Letrozole and (3) Everolimus + EDT in Patients With Unresectable Fibrolamellar Hepatocellular Carcinoma (FLL-HCC)',
   'sponsor': 'Memorial Sloan Kettering Cancer Center',
   'overall_status': 'COMPLETED',
   'last_known_status': None,
   'termination_reason': None,
   'phase': ['PHASE2'],
   'start_date': '2012-07-12',
   'primary_completion_date': '2021-07-16',
   'completion_date': '2021-07-16',
   'last_update_date': '2022-12-29',
   'is_us_export': None,
   'accepts_healthy_volunteers': False,
   'indication': ['Carcinoma: D002277', 'Carcinoma, Hepatocellular: D006528'],
   'treatment': ['Everolimus: D000068338',
    'Letrozole: D000077289',
    'Leuprolide: D016729']},
  {'id': 'NCT04025567',
   'title': 'Phase II Study of Oral Vancomycin in Patients With Unresectable Fibrolamellar Hepatocellular Carcinoma (FLC)'

### Helper functions for the extract_data function

In [32]:
def create_indication_and_treatment(clinical_trials: list[Clinical_Trial]) -> list:
    """Function that returns a list with 2 elements:
        1. List of Indication objects
        2. List of Treatment objects
    """
    final_result = []
    indications = []
    treatments = []
    for trial in clinical_trials:
        if len(trial.indication) != 0:
            for a in trial.indication:
                split = a.split(": ")
                indication = Indication(split[0], split[1]) # split[0] is the name, split[1] is the code
                indications.append(indication)
        if len(trial.treatment) != 0:
            for b in trial.treatment:
                split = b.split(": ")
                treatment = Treatment(split[0], split[1]) # split[0] is the name, split[1] is the code
                treatments.append(treatment)
    final_result.append(indications)
    final_result.append(treatments)
    return final_result

def remove_duplicates(objects: list) -> list:
    """Remove duplicates from a list of Indication or Treatment objects by ensuring we have no duplicate codes.
    """
    i = 0
    while i < len(objects):
        j = i + 1
        while j < len(objects):
            if objects[i].code == objects[j].code: # We are checking for duplicates by the indication or treatment code
                del objects[j]
            else:
                j += 1
        i += 1
    return objects

def serialize(objects: list) -> list:
    """Serialize a list of Clinical_Trial, Indication, or Treatment objects.
    """
    serialized_list = []
    for i in range(len(objects)):
        a = json.dumps(objects[i].__dict__) # json.dumps will return a JSON string
        serialized_list.append(json.loads(a)) # json.loads will convert the JSON string to a dictionary
    return serialized_list